In [25]:
#
# Collect the ACS 5 Year estimates for the variables in the myvars.csv
#

import geopandas as gpd
import fiona
import pandas as pd
import censusdata
import os
import pyproj

API_KEY = "UPDATE ME"
CENSUS_YEAR = 2020

# The Census api allows downloading all blockgroups
#     in a given state
#     up to 50 variables
# So, pull all block groups for one state at a time
#     make multiple requests for each state, split the variable list into 48 columns chunks (blockgroup ID and name are first)

myvars = pd.read_csv('acsvars.csv')

#states = censusdata.geographies(censusdata.censusgeo([('state', '*')]), 'acs5', 2021)
states= pd.read_csv("https://raw.githubusercontent.com/OpenEnvironments/core/refs/heads/main/states.csv")
states["StateFIPS"] = states["StateFIPS"].astype(str).str.zfill(2)
statelist = []

for state in states[(~states.CensusName.isna()) & (states.Abbreviation != 'PR')]["StateFIPS"]:
    print("Downloading",state,":",str(states[states.StateFIPS==state]["Possession"][0:1]))
    chunksize = 48
    thisstate = []
    for i in range(0, len(myvars.acsformat), chunksize):
        chunk = myvars.acsformat[i:i + chunksize]
        print(f"Batch {i // chunksize + 1}: items {i}-{i + len(chunk) - 1} ({len(chunk)} items)")
        thisstate.append(
            censusdata.download('acs5', CENSUS_YEAR, 
                                censusdata.censusgeo([('state', state), ('county', '*'), ('block group', '*')]),
                                ['GEO_ID']+list(chunk),
                                key=API_KEY,).reset_index().drop('index',axis=1)
        )
    # Now thisstate is a list of dataframes all with the same bg keys but different columns
    # Concatenate them horizontally and append the full state results to the master state list
    firstdf = True
    for d in thisstate:
        if firstdf:
            state_df = d
            firstdf = False
        else:
            state_df = pd.merge(state_df,d,on="GEO_ID",how="outer")
        print(state_df.shape)
    statelist.append(state_df)
acs = pd.concat(statelist)
acs['GEOID'] = acs.GEO_ID.str[9:] # the ACS GEO_ID is fully qualified, first 8 chars are meaningless here, eg "1500000US560459513001"
print('ACS collected')

tiger_folder = r"D:\Open Environments\data\census\tiger\2020 blockgroups"
tiger_files = []
for tiger_file in os.listdir(tiger_folder):
    print(tiger_file)
    tf = gpd.read_file(tiger_folder+"\\"+tiger_file)
    tf = tf.set_crs("EPSG:4269")
    tiger_files.append(tf)

tiger = pd.concat(tiger_files)

# tiger has more blockgroups than ACS because it includes Guam, Wake Island, American Samoa etc.
full = pd.merge(acs, tiger[['GEOID','ALAND', 'AWATER', 'INTPTLAT','INTPTLON',]], how='left', on='GEO_ID' )

full.to_csv('2020blockgroupdemographics.csv')

print("Merged dataset written")

Name: Possession, dtype: object
Batch 1: items 0-47 (48 items)
Batch 2: items 48-95 (48 items)
Batch 3: items 96-143 (48 items)


KeyboardInterrupt: 